In [0]:
%sql
-- 1- kpi AOV
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_aov AS
SELECT 
ROUND(COALESCE(SUM(revenue) / NULLIF(COUNT(DISTINCT order_id),0),0),2) AS aov
FROM de_mini_project.03_gold_schema.curated_orders;
-- 2-kpi Delivery Time 
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_delivery_time AS
SELECT ROUND(AVG(delivery_days),2) AS avg_days
FROM  de_mini_project.03_gold_schema.curated_orders;
-- 3-kpi KPI_clv 
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_clv AS
SELECT customer_id, SUM(revenue) clv
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY customer_id;
--4-kpi late-delivery 
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_delivery AS
SELECT ROUND(AVG(is_late_delivery)*100,2) AS late_delivery_percent
FROM de_mini_project.03_gold_schema.curated_orders;
-- 5-kpi low ratings
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_low_ratings AS
SELECT
SUM(CASE WHEN review_score < 3 THEN 1 ELSE 0 END)*100.0/COUNT(*) AS low_rating_pct
FROM de_mini_project.03_gold_schema.curated_orders;
-- 6 kpi customer new returning
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_customer_new_returning AS
WITH first_order AS (
SELECT customer_id, MIN(order_date) first_date
FROM  de_mini_project.03_gold_schema.curated_orders GROUP BY customer_id
)
SELECT
year, month,
COUNT(DISTINCT CASE WHEN order_date = first_date THEN c.customer_id END) new_customers,
COUNT(DISTINCT CASE WHEN order_date > first_date THEN c.customer_id END) returning_customers
FROM  de_mini_project.03_gold_schema.curated_orders c
JOIN first_order f ON c.customer_id=f.customer_id
GROUP BY year, month;
-- 7-kpi fulfillment
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_fulfillment AS
SELECT
COUNT(CASE WHEN order_status='delivered' THEN 1 END)*100.0/COUNT(*) AS fulfillment_rate
FROM de_mini_project.03_gold_schema.curated_orders;
-- 8 -kpi orders by location
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_orders_location AS
SELECT city, state, COUNT(DISTINCT order_id) total_orders
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY city, state;
-- 9 -kpi payment distribution
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_payment_distribution AS
SELECT 
payment_type,
COUNT(*) AS cnt,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS percentage
FROM de_mini_project.03_gold_schema.curated_orders
WHERE payment_type IS NOT NULL
GROUP BY payment_type;
-- 10 -kpi revenue monthly
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_revenue_monthly AS
SELECT year, month, ROUND(SUM(revenue),2) AS total_revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY year, month;
-- 11 -kpi revenue by product
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_reviews_product AS
SELECT product_category_name, ROUND(AVG(review_score),2) avg_rating
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY product_category_name;
-- 12 -kpi revenue by seller
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_reviews_seller AS
SELECT seller_id, ROUND(AVG(review_score),2) as avg_rating
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY seller_id;
-- 13 -kpi top products
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_top_products AS
SELECT product_category_name, SUM(revenue) AS revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY product_category_name
ORDER BY revenue DESC
LIMIT 10;
-- 14 -kpi top seller
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_top_sellers AS
SELECT seller_id, ROUND(SUM(revenue),2) revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY seller_id
ORDER BY revenue DESC LIMIT 10;